In [1]:
import sys

# Point to your project root (folder that contains smart_stethoscope/)
project_path = "/Users/keira/code/mi-mi-mia/smart-stethoscope"

if project_path not in sys.path:
    sys.path.append(project_path)

print("Path added ✅")

Path added ✅


In [2]:
from smart_stethoscope.interface.main import train

# Run full training pipeline
results = train()

# Extract models
xgb_model = results["xgb_model"]
cnn_model = results["cnn_model"]

# Print sanity checks
print("Train samples:", len(results["train_idx"]))
print("Val samples:", len(results["val_idx"]))

print("X shape:", results["X"].shape)
print("Mel shape:", results["mel_spec"].shape)

print("Done ✅")


Skipped blacklisted patient 142

Skipped blacklisted patient 182

Skipped blacklisted patient 191

Skipped blacklisted patient 191

Skipped blacklisted patient 191
Train classes: [0 1 2 3 4]
Val classes:   [0 1 2 3 4]
Train X shape: (2011, 42)
Val X shape:   (1124, 42)
Train mel shape: (2011, 128, 141, 1)
Val mel shape:   (1124, 128, 141, 1)


2026-03-25 16:15:39.613473: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M4
2026-03-25 16:15:39.613498: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 16.00 GB
2026-03-25 16:15:39.613510: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 5.33 GB
2026-03-25 16:15:39.613525: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2026-03-25 16:15:39.613534: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)
2026-03-25 16:15:40.709659: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.


Train samples: 2011
Val samples: 1124
X shape: (3135, 42)
Mel shape: (3135, 128, 141, 1)
Done ✅


In [3]:
import joblib, json, os
from smart_stethoscope.params import CLASS_NAMES

model_dir = "models"
os.makedirs(model_dir, exist_ok=True)

# Save models
joblib.dump(results["xgb_model"], f"{model_dir}/xgb_model.pkl")
results["cnn_model"].save(f"{model_dir}/cnn_model.keras")

# Save feature columns
joblib.dump(results["X"].columns.tolist(), f"{model_dir}/feature_columns.pkl")

# Save class names (UPDATED + correct order)
with open(f"{model_dir}/class_names.json", "w") as f:
    json.dump(CLASS_NAMES, f)

print("Clean model package saved ✅")

Clean model package saved ✅


In [4]:
import joblib
from tensorflow.keras.models import load_model
import json

xgb = joblib.load("models/xgb_model.pkl")
cnn = load_model("models/cnn_model.keras")
cols = joblib.load("models/feature_columns.pkl")

with open("models/class_names.json") as f:
    class_names = json.load(f)

print("All assets load ✅")
print(class_names)

All assets load ✅
['COPD', 'Pneumonia', 'Healthy', 'URTI', 'Bronchiectasis']


In [5]:
import librosa as lb
from smart_stethoscope.interface import main

# Load ONE audio file (your example)
audio, sr = lb.load(
    "/Users/keira/code/mi-mi-mia/smart-stethoscope/raw_data/Respiratory_Sound_Database/Respiratory_Sound_Database/audio_and_txt_files/101_1b1_Al_sc_Meditron.wav"
)

# Preprocess that one recording
features, mel_spectograms = main.preprocess_for_prediction(
    audio=audio,
    sampling_rate=sr,
    start=0.036,
    end=19.964
)

# Run prediction
preds = main.predict(
    xgb_model=results["xgb_model"],
    cnn_model=results["cnn_model"],
    xgb_features=features,
    cnn_features=mel_spectograms
)

# Extract final result
final_pred = preds["final_prediction"]

print("Prediction:", final_pred["predicted_label"])
print("Confidence:", final_pred["confidence"])
print("Class probabilities:", final_pred["class_probabilities"])

Prediction: Bronchiectasis
Confidence: 0.5472423434257507
Class probabilities: {'COPD': 0.03784500062465668, 'Pneumonia': 0.1993626207113266, 'Healthy': 0.11611894518136978, 'URTI': 0.09943107515573502, 'Bronchiectasis': 0.5472423434257507}
